In [1]:
import numpy as np
import pandas as pd

Набор данных `clothes.xlsx` содержит сведения о работе предприятия по производству лёгкой одежды. На листе "Ткани" представлены виды тканей, используемых при пошиве; на листе "Продукция" - информация о моделях выпускаемой одежды; на листе "Готовый товар" - информация об уже произведённой фирмой одежде.

In [4]:
fabrics = pd.read_excel("clothes.xlsx", sheet_name="Ткани")
products = pd.read_excel("clothes.xlsx", sheet_name="Продукция")
finished = pd.read_excel("clothes.xlsx", sheet_name="Готовый товар")

In [9]:
print(
  "ТКАНИ", fabrics.head(),
  "ПРОДУКЦИЯ", products.head(),
  "ГОТОВЫЙ ТОВАР", finished.head(),
  sep="\n\n"
)

ТКАНИ

  ID ткани Название     Цвет  Плотность, г/м2   Сырье  Ширина полотна, см
0       T1    атлас  красный              240    шёлк                 150
1       T2    атлас   желтый              240    шёлк                 150
2       T3    атлас    синий              240    шёлк                 150
3       T4   бархат  красный              405  хлопок                 150
4       T5   бархат    синий              405  хлопок                 150

ПРОДУКЦИЯ

  ID товара Наименование товара Размерный ряд  Расход материала, см  \
0        P1     юбка полусолнце       110-152                    70   
1        P2      юбка с запахом       110-152                    90   
2        P3   юбка со складками       110-152                    90   
3        P4         юбка солнце       110-152                   100   
4        P5              бриджи       110-152                   160   

  Категория потребителей  
0                девочки  
1                девочки  
2                девочки  
3 

In [13]:
# Внутреннее соединение всех трёх таблиц
merged = finished.merge(products, on="ID товара", how="inner") \
                 .merge(fabrics, on="ID ткани", how="inner")
merged.head()

,Артикул,ID товара,ID ткани,"Количество на складе, шт","Отпускная цена, руб.",Наименование товара,Размерный ряд,"Расход материала, см",Категория потребителей,Название,Цвет,"Плотность, г/м2",Сырье,"Ширина полотна, см"
0,A1,P22,T23,32,700,юбка солнце,44-52,120,женщины,крепдешин,красный,196,шёлк,140
1,A2,P59,T13,70,700,бермуды,44-54,240,мужчины,вельвет,красный,410,хлопок,150
2,A3,P15,T31,43,701,брюки прямые,110-152,200,девочки,лён,желтый,160,лён,150
3,A4,P48,T22,112,701,платье ретро,44-52,235,женщины,крепдешин,синий,196,шёлк,140
4,A5,P11,T24,32,702,платье прямое,110-152,180,девочки,крепдешин,зеленый,196,шёлк,140


In [ ]:
# 1. Выполните внутреннее соединение данных в один датафрейм. Укажите кол-во получившихся строк.
print("1:", len(merged))

1: 871


In [ ]:
# 2. Укажите общее кол-во единиц товара на складе.
print("2:", finished["Количество на складе, шт"].sum())

2: 53828


In [14]:
# 3. Используя информацию из приведённого набора данных,
# определите общую стоимость (в рублях) всех красных платьев,
# произведённых на предприятии из хлопковой ткани плотностью не менее 100 г/м2.
mask3 = (
  merged["Наименование товара"].str.contains("платье", case=False) &
  (merged["Цвет"] == "красный") &
  (merged["Сырье"] == "хлопок") &
  (merged["Плотность, г/м2"] >= 100)
)
total_cost = (merged.loc[mask3, "Количество на складе, шт"] *
              merged.loc[mask3, "Отпускная цена, руб."]).sum()
print("3:", total_cost)

3: 2461397


In [15]:
# 4. Укажите среднюю цену красного платья из бязи для девочек.
mask4 = (
  merged["Наименование товара"].str.contains("платье", case=False) &
  (merged["Цвет"] == "красный") &
  (merged["Название"] == "бязь") &
  (merged["Категория потребителей"] == "девочки")
)
print("4:", merged.loc[mask4, "Отпускная цена, руб."].mean())

4: 1001.0


In [16]:
# 5. Определите кол-во прямых брюк на складе для женщин.
mask5 = (
  merged["Наименование товара"].str.contains("брюки прямые", case=False) &
  (merged["Категория потребителей"] == "женщины")
)
print("5:", merged.loc[mask5, "Количество на складе, шт"].sum())

5: 1286


In [23]:
# 6. Укажите название ткани, которая пока не использовалась предприятием для пошива одежды.
used_fabric_ids = finished["ID ткани"].unique()
unused = fabrics[~fabrics["ID ткани"].isin(used_fabric_ids)]
print("6:", unused["Название"].values)

6: <StringArray>
['полиэстер']
Length: 1, dtype: str


In [24]:
# 7. Укажите кол-во товаров, которых нет на складе (в таблице "Готовый товар").
print("7:", (finished["Количество на складе, шт"] == 0).sum())

7: 0


In [25]:
# 8. Укажите наименование товара с самым большим расходом материала.
idx8 = products["Расход материала, см"].idxmax()
print("8:", products.loc[idx8, "Наименование товара"])

8: платье-жилет


In [26]:
# 9. Какая ткань обладает наибольшей плотностью?
idx9 = fabrics["Плотность, г/м2"].idxmax()
print("9:", fabrics.loc[idx9, "Название"])

9: драп


In [27]:
# 10. Сколько всего различных видов продукции представлено на складе?
in_stock = finished[finished["Количество на складе, шт"] > 0]
print("10:", in_stock["ID товара"].nunique())

10: 45
